<a href="https://colab.research.google.com/github/Ewanjohndennis/flyrankml/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I'm choosing **Lane 2: Refresh / Content Opportunity Scoring**.

The core question is: given a large content inventory, which pages should a reviewer look at first?

This lane fits because:
- The starter pipeline already runs end-to-end on this problem, giving me a concrete baseline to understand and improve on.
- The output is a ranked list with reason codes — something a real human reviewer can actually act on.
- The gap between the baseline rule and the learned model is large enough to be worth understanding (more on that below).

My capstone goal is not to just re-run the starter. It's to understand *why* the model beats the baseline, define a stronger label using the full warehouse (future-window decline instead of current-window trend), and validate honestly against held-out clients.

## 2. The question: decision, action, cost of a wrong call

**Research question:**  
Which content pages should be prioritized for review and refresh, based on observable search and engagement signals?

**Unit of analysis:** One page (one content item). Each row in the output represents one page with a score, a suggested action, and reason codes explaining the score.

**The decision this improves:**  
A content reviewer has limited time. They cannot look at every page. This system helps them spend their time on the pages most likely to benefit from attention — rather than picking randomly or going by gut feel.

**The action someone takes from it:**  
Look at the top-ranked pages first. For each one, the reason codes tell you *why* it was flagged — declining traffic, low CTR despite good position, thin content, stale age. The reviewer then decides whether to refresh, expand, or leave it alone.

**Cost of a wrong recommendation:**  
- A false positive (flagging a healthy page): wasted reviewer time, low cost.
- A false negative (missing a page that actually needed attention): the page keeps losing traffic, the opportunity is missed.
- The bigger risk here is false negatives on high-traffic pages — missing a page with 10,000+ impressions that is quietly declining is more costly than flagging a page that turns out fine.

**Why data and ML help here:**  
A fixed rule (like "flag pages older than 180 days with 500+ impressions") captures some signal but ignores the interaction between signals. A page that is old, declining, AND has low CTR for its position is more urgent than a page that is merely old. A learned model can weigh combinations of signals that a hand-written rule misses.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [2]:
import pandas as pd

# Download the starter dataset from the repo
url = "https://raw.githubusercontent.com/Ewanjohndennis/flyrankml/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print('=== Dataset overview ===')
print(f'Total pages:                {len(df):,}')
print(f'Pages with declining trend: {(df["trend_direction"] == "down").sum():,}')
print(f'Declining rate:             {(df["trend_direction"] == "down").mean():.1%}')
print()

print('=== Number 1: Baseline rule is weak ===')
print('Precision@50 — how many of the top 50 picks are real targets:')
print('  Baseline rule:  0.240  (~12 correct out of 50)')
print('  Random forest:  0.740  (~37 correct out of 50)')
print('  A fixed rule misses most of the best candidates.')
print()

print('=== Number 2: Most important signal is not obvious ===')
print('Top feature by importance: days_with_impressions (0.158)')
print('This is search visibility consistency over time — not age or word count.')
print('A hand-written rule based on age alone would miss this.')
print()

print('=== Number 3: Impression gap between declining and stable pages ===')
declining = df[df['trend_direction'] == 'down']['impressions_90d']
stable    = df[df['trend_direction'] != 'down']['impressions_90d']
print(f'Median impressions — declining pages: {declining.median():,.0f}')
print(f'Median impressions — stable pages:    {stable.median():,.0f}')
print('Declining pages are not just low-traffic noise — they have real search volume.')

=== Dataset overview ===
Total pages:                30,000
Pages with declining trend: 16,262
Declining rate:             54.2%

=== Number 1: Baseline rule is weak ===
Precision@50 — how many of the top 50 picks are real targets:
  Baseline rule:  0.240  (~12 correct out of 50)
  Random forest:  0.740  (~37 correct out of 50)
  A fixed rule misses most of the best candidates.

=== Number 2: Most important signal is not obvious ===
Top feature by importance: days_with_impressions (0.158)
This is search visibility consistency over time — not age or word count.
A hand-written rule based on age alone would miss this.

=== Number 3: Impression gap between declining and stable pages ===
Median impressions — declining pages: 961
Median impressions — stable pages:    472
Declining pages are not just low-traffic noise — they have real search volume.


## 4. Careful words: what I can and can't claim

**What I can claim, if validated:**
- Observable signals (search visibility consistency, impressions, position, CTR) are associated with whether a page is in a declining trend.
- A learned model, validated on held-out clients, can rank declining pages higher than a fixed rule.
- The ranked output helps a reviewer prioritize their limited time.

**What I cannot claim:**
- That refreshing a flagged page will cause traffic to recover. This data shows association, not causation. Proving causation would need a controlled experiment.
- That the starter label (`trend_direction == 'down'`) is the ideal target. It is a current-window proxy. A stronger label uses features from the prior 90 days to predict decline over the next 30 days — that is what the capstone will work toward.
- That results from the 30,000-row starter slice will hold exactly on the full warehouse (~519,000 content items, ~79M daily rows). The scale and client mix are different, and the result has to be earned again.
- That I have identified Google ranking factors. These are internal content and search signals — not evidence about how Google's algorithm works.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.